In [1]:
import os
import json
from dotenv import load_dotenv
from langchain_community.chat_models import ChatOpenAI
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Pinecone as PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import ChatPromptTemplate, PromptTemplate

/Users/koterukhyathi/Desktop/GenAI/rag-project/.venv/lib/python3.8/site-packages/pinecone/data/index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
import sys
print(sys.executable)
import tiktoken
print("tiktoken OK")

/Users/koterukhyathi/Desktop/GenAI/rag-project/.venv/bin/python
tiktoken OK


In [3]:
load_dotenv(dotenv_path=".env", override=True)

True

In [4]:
import os
print('Notebook cwd:', os.getcwd())
print('OPENAI_API_KEY present:', bool(os.getenv('OPENAI_API_KEY')))
print('PINECONE_API_KEY present:', bool(os.getenv('PINECONE_API_KEY')))
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Missing OPENAI_API_KEY — set it in .env or environment and restart the kernel')


Notebook cwd: /Users/koterukhyathi/Desktop/GenAI/rag-project
OPENAI_API_KEY present: True
PINECONE_API_KEY present: True


In [5]:
import sys, importlib
print('kernel executable:', sys.executable)
print('python version:', sys.version)
print('sys.path sample:', sys.path[:5])
try:
    import tiktoken
    print('tiktoken import OK, module file:', getattr(tiktoken, '__file__', 'n/a'))
except Exception as e:
    print('tiktoken import failed:', type(e).__name__, e)
    raise


kernel executable: /Users/koterukhyathi/Desktop/GenAI/rag-project/.venv/bin/python
python version: 3.8.3 (default, Jul  2 2020, 11:26:31) 
[Clang 10.0.0 ]
sys.path sample: ['/opt/anaconda3/lib/python38.zip', '/opt/anaconda3/lib/python3.8', '/opt/anaconda3/lib/python3.8/lib-dynload', '', '/Users/koterukhyathi/Desktop/GenAI/rag-project/.venv/lib/python3.8/site-packages']
tiktoken import OK, module file: /Users/koterukhyathi/Desktop/GenAI/rag-project/.venv/lib/python3.8/site-packages/tiktoken/__init__.py


In [6]:
# Load the knowledge base from JSON and wrap each entry in a Document.
with open("passenger_assistance.json") as f:
    kb = json.load(f)



In [7]:
def profile_to_sections(p):
    return [
        {
            "section": "identity",
            "content": (
                f"SSR code {p['ssr_code']}: {p['disability_category']}. "
                f"{p['description']}"
            )
        },
        {
            "section": "services",
            "content": (
                f"Allowed services for {p['ssr_code']}: "
                f"{', '.join(p['allowed_services'])}. "
                f"Excluded services: "
                f"{', '.join(p['excluded_services']) if p['excluded_services'] else 'none'}. "
                f"Seating restrictions: "
                f"{', '.join(p.get('seating_restrictions', ['none']))}."
            )
        },
        {
            "section": "compliance",
            "content": (
                f"Compliance for {p['ssr_code']}: "
                f"Regulations: {', '.join(p['regulatory_citations'])}. "
                f"Advance notice: {p.get('advance_notice_hours', 0)} hours. "
                f"SLA: {p.get('sla_pickup_minutes', 'n/a')} minutes. "
                f"Notes: {p.get('notes', '')}."
            )
        }
    ]

profile_docs = [
    Document(
        page_content=sec["content"],
        metadata={
            "profile_id":           p["profile_id"],
            "ssr_code":             p["ssr_code"],
            "section":              sec["section"],   # identity | services | compliance
            "doc_type":             "profile",
            "advance_notice_hours": p.get("advance_notice_hours", 0),
            "sla_minutes":          p.get("sla_pickup_minutes", 0),
        }
    )
    for p in kb["passenger_profiles"]
    for sec in profile_to_sections(p)
]

rule_docs = [
    Document(
        page_content=f"Rule {r['rule_id']}: {r['rule']} Citation: {r['citation']}",
        metadata={"doc_type": "rule", "rule_id": r["rule_id"]}
    )
    for r in kb["business_rules"]
]

service_docs = [
    Document(
        page_content=f"Service {name}: {desc}",
        metadata={"doc_type": "service", "service_code": name}
    )
    for name, desc in kb["service_catalog"].items()
]

all_docs = profile_docs + rule_docs + service_docs
print(f"Profile chunks : {len(profile_docs)}")   # 36
print(f"Rule chunks    : {len(rule_docs)}")       # 5
print(f"Service chunks : {len(service_docs)}")    # 20
print(f"Total          : {len(all_docs)}") 

Profile chunks : 36
Rule chunks    : 5
Service chunks : 20
Total          : 61


In [8]:
profile_docs[0].metadata

{'profile_id': 'DP-001',
 'ssr_code': 'WCHR',
 'section': 'identity',
 'doc_type': 'profile',
 'advance_notice_hours': 0,
 'sla_minutes': 20}

Indexing

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)

/var/folders/6g/d767h6sx71n4_99dgkfp_q5m0000gn/T/ipykernel_46664/2933090828.py:2: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512, openai_api_key=os.getenv('OPENAI_API_KEY'))
/Users/koterukhyathi/Desktop/GenAI/rag-project/.venv/lib/python3.8/site-packages/langchain_community/embeddings/openai.py:270: UserWarning: WARNING! dimensions is not default parameter.
                    dimensions was transferred to model_kwargs.
                    Please confirm that dimensions is what you intended.
  warnings.warn(


In [10]:
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

INDEX_NAME = "wheelchair-assistance"

# Only create if it doesn't exist
if INDEX_NAME not in [i.name for i in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,
        dimension=512,          # must match text-embedding-3-small
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"   # change to your Pinecone region
        )
    )
    print(f"Index '{INDEX_NAME}' created.")
else:
    print(f"Index '{INDEX_NAME}' already exists.")


Index 'wheelchair-assistance' already exists.


In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)

# This embeds all_docs and upserts into Pinecone in one call
vectorstore = PineconeVectorStore.from_documents(
    documents=all_docs,           # same all_docs from chunking step
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace="wheelchair-assistance"
)

print(f"Upserted {len(all_docs)} chunks into Pinecone.")

Upserted 61 chunks into Pinecone.


In [ ]:
# vectorstore = PineconeVectorStore.from_existing_index(
#     index_name=INDEX_NAME,
#     embedding=embeddings,
# )


In [13]:
filter_dict = {"doc_type": {"$eq": "profile"}}
question = "Can a WCHC passenger use an aisle chair?"

Generator

In [21]:
#configure the llm
llm = ChatOpenAI(model="gpt-4.1-mini")

#set the prompt template
template = """You are a wheelchair assistance operations expert at an airport.
Answer using ONLY the context below.
Cite the regulation when relevant.
If a service is EXCLUDED, say so clearly.

Context:
{context}

Customer issue: {question}

Support guidance:"""

rag_prompt_template = PromptTemplate.from_template(template)

In [22]:
#generated response
def generate_answer(user_question):
    #retrieve the relevant docs
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5}, 
                                          search_type='similarity',
                                          filter=filter_dict)
    retrieved_docs = retriever.invoke(question)

    #generate
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = rag_prompt_template.invoke({"question": user_question, "context": docs_content})
    response = llm.invoke(prompt)

    return response.content

generate_answer("Can a WCHC passenger use an aisle chair?")

'Yes, a WCHC passenger can use an aisle chair. The service AISLE_CHAIR_TRANSFER is explicitly allowed for WCHC passengers and is defined as onboard aisle chair use for boarding, deplaning, and lavatory access. This aligns with the SSR code WCHC indicating the passenger is completely immobile and requires assistance from check-in to the aircraft seat, including use of an aisle chair.'

In [23]:
generate_answer("Does a blind passenger get a wheelchair")

'A blind passenger is not automatically eligible for a wheelchair service based solely on blindness. Wheelchair services (WCHS, WCHC, WCHR) are designated based on mobility needs, not vision impairment.\n\n- If the passenger has a mobility impairment (e.g., completely immobile), they may qualify for WCHC services, which include wheelchair assistance from check-in to seat, aisle chair transfer, and additional accommodations per 14 CFR 382.81.\n- If the passenger only has a visual impairment, they are typically not provided wheelchair services unless they also request and require assistance related to mobility.\n\nTherefore, a blind passenger does **not automatically receive a wheelchair** unless they also have a mobility impairment qualifying them for a service such as WCHC or WCHS.\n\nFor any passenger, assistance such as CONNECTION_ESCORT or SECURITY_ESCORT is allowed, which can help with navigation but does not involve wheelchair provision unless mobility assistance is needed.\n\nIn 

In [19]:
generate_answer("can we charge for wheelchair assistance?")

'Wheelchair assistance services (WCHS, WCHC, WCHR) are typically provided as part of required disability accommodations and should not be charged to the passenger. Therefore, you should not charge for wheelchair assistance. If further billing questions arise, refer to company policy on disability accommodations.'

In [1]:
!pip install ragas

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from datasets import Dataset

eval_data = {
    "question": [
        "Can a WCHC passenger use an aisle chair?",
        "Can we charge for wheelchair assistance?",
    ],
    "answer": [
        ask("Can a WCHC passenger use an aisle chair?"),
        ask("Can we charge for wheelchair assistance?"),
    ],
    "contexts": [
        [retrieve_policy("Can a WCHC passenger use an aisle chair?")],
        [retrieve_policy("Can we charge for wheelchair assistance?")],
    ],
    "ground_truth": [
        "Yes, AISLE_CHAIR_TRANSFER is in allowed services for WCHC.",
        "No. Airlines cannot charge under 14 CFR Part 382.",
    ]
}

results = evaluate(Dataset.from_dict(eval_data),
                   metrics=[faithfulness, answer_relevancy, context_precision])
print(results)


/Users/koterukhyathi/Desktop/GenAI/rag-project/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValidationError: 1 validation error for OpenAIEmbeddings
openai_api_key
  extra fields not permitted (type=value_error.extra)